# FLEO-FER on Kaggle GPU

Train the FLEO-augmented YOLOv12-S emotion detector on FER2013 + RAF-DB, compute the fold-out delta, and export R1/R2/R3 ONNX for Vitis AI / ZCU104.

**Settings (right sidebar):** Accelerator = GPU P100, Internet = ON.  
**Add Input:** `msambare/fer2013` and `shuvoalok/raf-db-dataset`.

## 1. Clone + install (do NOT reinstall torch — Kaggle ships CUDA torch)

In [ ]:
!git clone https://github.com/olfa-askri/FLEO.git
%cd FLEO
!pip install -q ultralytics onnx onnxruntime onnxscript
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))

## 2. Prepare datasets (auto-detects layout)

In [ ]:
!python -m data.prepare_fer2013 --src /kaggle/input/fer2013        --out datasets/fer2013
!python -m data.prepare_rafdb   --src /kaggle/input/raf-db-dataset --out datasets/rafdb
!cat datasets/fer2013/data.yaml

## 3. (Optional) 1-minute smoke test on synthetic data before spending quota

In [ ]:
!python -m data.make_synthetic --out datasets/synthetic
!python -m scripts.train --data datasets/synthetic/data.yaml --variant fleo --epochs 2 --imgsz 96 --batch 8 --device 0 --workers 2

## 4. Full matrix per dataset (start with 1 seed; use `--seeds 0 1 2` for the paper's mean±sd)

In [ ]:
!python -m scripts.run_matrix --data datasets/fer2013/data.yaml --dataset fer2013 \n    --seeds 0 --epochs 100 --imgsz 160 --batch 64 --device 0

In [ ]:
!python -m scripts.run_matrix --data datasets/rafdb/data.yaml --dataset rafdb \n    --seeds 0 --epochs 100 --imgsz 160 --batch 64 --device 0

## 5. Export the three deployment routes (FP32 ONNX for Vitis AI)

In [ ]:
W='runs/fleo/fleo_seed0/weights/best.pt'
!python -m scripts.export --weights {W} --route r1 --imgsz 160 --verify
!python -m scripts.export --weights {W} --route r2 --imgsz 160 --verify
!python -m scripts.export --weights {W} --route r3 --imgsz 160
!ls -la export

## 6. Bundle artifacts to download (Output tab) or Save Version to persist

In [ ]:
!cd /kaggle/working/FLEO && zip -qr /kaggle/working/fleo_artifacts.zip export results runs/fleo/*/weights/best.pt
print('done -> /kaggle/working/fleo_artifacts.zip')